In [1]:
import os
from distutils.dir_util import copy_tree
import pandas as pd
import numpy as np

In [2]:
# Urbanisation degrees in this study
# The following folders contain the raw data for the reference case
dirs = {2: '2021-05-19_114409_simbev_run_SR_Mitte',
        3: '2021-05-19_114212_simbev_run_LR_Klein'}

In [3]:
# Sufficiency scenarios quantified with quetzal_germany
# https://github.com/marlinarnz/quetzal_germany/releases/tag/v2.1.0
scenarios = ['Avoid', 'Shift', 'Avoid+Shift']
scenario_ref = 'reference'

# Step 1) Car ownership

Delete every Xth driving pattern, depending on how much reduction in car availability per household is seen in the corresponding scenario for the corresponding region (compared to the reference scenario).

In [4]:
# Get car availability from scenarios
car_av = pd.DataFrame(index=list(dirs.keys()),
                      data={'reference': [0.92855, 0.94875],
                            'Avoid': [0.717309, 0.838803],
                            'Shift': [0.906636, 0.9361267],
                            'Avoid+Shift': [0.5460803, 0.7392621]})
car_av.T - car_av['reference']

,2,3
reference,0.000000,0.000000
Avoid,-0.211241,-0.109947
Shift,-0.021914,-0.012623
Avoid+Shift,-0.382470,-0.209488


In [5]:
for scenario in scenarios:
    # Create new directories with charging profiles and copy them
    # NOTE: you must delete the old scenario directory first
    for u in dirs.keys():
        os.makedirs(scenario+'/'+dirs[u])
        copy_tree(os.path.normpath(scenario_ref+'/'+dirs[u]+'/'),
                  os.path.normpath(scenario+'/'+dirs[u]+'/'))
    
    # Delete every Xth car
    for u in dirs.keys():
        files = os.listdir(scenario+'/'+dirs[u])
        step = round(1 / -(car_av.T - car_av['reference']).loc[scenario, u])
        for i in range(0, len(files), step):
            os.remove(scenario+'/'+dirs[u]+'/'+files[i])
        print('{}, u={}: {} cars'.format(scenario, u, len(os.listdir(scenario+'/'+dirs[u]))))

Avoid, u=2: 96 cars
Avoid, u=3: 106 cars
Shift, u=2: 117 cars
Shift, u=3: 118 cars
Avoid+Shift, u=2: 80 cars
Avoid+Shift, u=3: 96 cars


In [6]:
# Calculate the difference between the number of profiles and
# the real number applying above's car availability shares
for scenario in scenarios:
    for u in dirs.keys():
        files = os.listdir(scenario+'/'+dirs[u])
        diff = (car_av.T - car_av['reference']).loc[scenario, u]
        step = round(1 / -diff)
        rest = ((1 - len(files) / 120) + diff) * len(files)
        print('{}, u={}: {}'.format(scenario, u, round(rest)))

Avoid, u=2: -1
Avoid, u=3: 1
Shift, u=2: 0
Shift, u=3: 0
Avoid+Shift, u=2: -4
Avoid+Shift, u=3: -1


# Step 2) Reduce trips and distance by purpose

Within the remaining charging profiles, continue for each trip purpose (commuting, business, education, shopping, leisure, accompany):
* Delete every Xth trip for the corresponding purpose based on the difference in trip frequency with private cars
* Reduce the trip distance (i.e. charging demand) by the difference in average distance of the scenario and region, compared to the reference

In [7]:
# Map sufficiency scenario demand segments to profile purposes
purposes = {'commuting_car': '0_work',
            'business_car': '1_business',
            'education_car': '2_school',
            'buy/execute_car': '3_shopping',
            'leisure_car': '5_leisure',
            'accompany_car': '4_private/ridesharing'}

In [9]:
# Load data from modelling results
if os.path.exists('car_use_results_scenarios.csv'):
    car_use = pd.read_csv('car_use_results_scenarios.csv', index_col=['urbanisation', 'purpose', 'unit'])
    
else:
    # Generate the results from model files
    import geopandas as gpd
    import shapely
    from shapely import speedups
    from tqdm import tqdm
    from quetzal.model import stepmodel
    
    # Load model zones
    model_path = '../quetzal_germany/model/'
    sm = stepmodel.read_json(model_path + scenario_ref + '/de_zones')
    zones = gpd.GeoDataFrame(sm.zones, crs=sm.epsg)
    scenario_results = {}
    # For results aggregation
    agg_dict_sum = {segment: 'sum' for segment in purposes.keys()}
    agg_dict_sum.update({segment+'_pkm': 'sum' for segment in purposes.keys()})
    agg_dict_sum['urbanisation'] = 'first'
    agg_dict_mean = {segment: 'mean' for segment in purposes.keys()}
    agg_dict_mean.update({segment+'_pkm': 'mean' for segment in purposes.keys()})
    
    # Load assigned networks
    '''
    shapely.speedups.enable()
    for s in [scenario_ref] + scenarios:
        sm = stepmodel.read_zippedpickles(model_path + s + '/de_assignment')
        sm.road_links = gpd.GeoDataFrame(sm.road_links.loc[sm.road_links['volume']>1], crs=sm.epsg)
        # Map origin zone
        road_geo = gpd.GeoDataFrame({'origin': np.nan},
                                    index=sm.road_links.index,
                                    geometry=[shapely.geometry.Point(g.coords[0])
                                              for g in sm.road_links['geometry']],
                                    crs=sm.epsg)
        for zone, geo in tqdm(zones['geometry'].items(), total=len(zones)):
            road_geo.loc[road_geo['geometry'].within(geo), 'origin'] = zone
        sm.road_links['origin'] = road_geo['origin']
        sm.road_links = sm.road_links.loc[sm.road_links['origin'].notna()]
        # Add urbanisation and pkm
        sm.road_links['urbanisation'] = sm.road_links['origin'].map(zones['urbanisation']).astype(int)
        for seg in purposes.keys():
            sm.road_links[seg+'_pkm'] = sm.road_links[seg] * sm.road_links['length'] / 1000
        
        # Aggregate results by urbanisation degree
        # Take the sum of zones and calculate the mean by urbanisation degree
        scenario_results[s] = sm.road_links.loc[sm.road_links['urbanisation'].isin(dirs.keys())]\
                              .groupby('origin').agg(agg_dict_sum)\
                              .groupby('urbanisation').agg(agg_dict_mean)
        del sm'''
    
    # Load level-of-service tables
    model_path = '../quetzal_germany/model/'
    for s in [scenario_ref] + scenarios:
        sm = stepmodel.read_zippedpickles(model_path + s + '/de_road_los')
        v = stepmodel.read_zippedpickles(model_path + s + '/de_volumes')
        sm.volumes = v.volumes
        sm.volumes.drop('index', axis=1, inplace=True, errors='ignore')
        sm.los = sm.car_los
        sm.segments = list(purposes.keys())
        # Compute volumes and pkm
        sm.compute_los_volume(keep_segments=True)
        for seg in purposes.keys():
            sm.los[seg+'_pkm'] = sm.los[seg] * sm.los['length'] / 1000
        # Aggregate results
        # Take the sum of zones and calculate the mean by urbanisation degree
        sm.los['urbanisation'] = sm.los['origin'].map(zones['urbanisation'])
        scenario_results[s] = sm.los.loc[sm.los['urbanisation'].isin(dirs.keys())]\
                              .groupby('origin').agg(agg_dict_sum)\
                              .groupby('urbanisation').agg(agg_dict_mean)
        del sm
    
    # Save results
    car_use = pd.DataFrame({s: scenario_results[s].T.stack() for s in scenario_results.keys()})
    car_use.index.names = ['purpose', 'urbanisation']
    car_use.reset_index(inplace=True)
    car_use['unit'] = ['volume'] * len(purposes)*2 + ['pkm'] * len(purposes)*2
    car_use['purpose'] = car_use['purpose'].apply(lambda s: s.split('_')[0])
    car_use = car_use.groupby(['urbanisation', 'purpose', 'unit']).sum()
    car_use.to_csv('car_use_results_scenarios.csv')

In [10]:
(car_use / 1e6).round(2)

reference  Avoid  Shift  Avoid+Shift
urbanisation purpose     unit                                        
2            accompany   pkm         23.60  16.45  18.36         9.33
                         volume       0.78   0.54   0.60         0.30
             business    pkm         63.14  25.43  47.59         1.89
                         volume       0.42   0.19   0.19         0.04
             buy/execute pkm         72.06  41.05  44.60        19.79
                         volume       2.50   1.41   1.56         0.67
             commuting   pkm        167.43  64.26  55.95        19.82
                         volume       3.09   1.54   1.27         0.55
             education   pkm          6.81   5.26   3.26         2.32
                         volume       0.25   0.21   0.11         0.08
             leisure     pkm        137.20  60.63  92.89        33.27
                         volume       2.74   1.22   1.75         0.64
3            accompany   pkm         10.81   8.01   9.76         5.65
                         volume       0.35   0.26   0.31         0.18
             business    pkm         22.69  13.33  27.93         1.71
                         volume       0.18   0.10   0.13         0.04
             buy/execute pkm         31.88  20.08  22.41        11.67
                         volume       1.08   0.69   0.76         0.39
             commuting   pkm         69.15  40.49  39.07        18.05
                         volume       1.34   1.00   0.89         0.51
             education   pkm          3.67   2.80   2.13         1.48
                         volume       0.13   0.10   0.06         0.04
             leisure     pkm         56.23  25.99  44.19        17.50
                         volume       1.06   0.50   0.76         0.31

In [11]:
# Modify charging patterns
for scenario in scenarios:
    for u in dirs.keys():
        car_use_ref = car_use[scenario_ref].xs(u, level='urbanisation')
        path = scenario+'/'+dirs[u]+'/'
        charging_demand = []
        
        # Take each pattern
        for file in os.listdir(path):
            profile = pd.read_csv(path + file, index_col=0).reset_index(drop=True)
            # For each purpose
            for p_quetzal, p_simbev in purposes.items():
                n_trips_ref = len(profile.loc[profile['location']==p_simbev])
                car_use_diff = car_use[scenario].xs(u, level='urbanisation')\
                               .xs(p_quetzal.split('_')[0], level='purpose') / \
                               car_use_ref.xs(p_quetzal.split('_')[0], level='purpose')
                
                # Delete the corresponding share of trips
                # Find stays to drop
                n_trips = round(n_trips_ref * car_use_diff['volume'])
                if n_trips <= n_trips_ref:
                    purpose = profile.loc[profile['location']==p_simbev].index
                    to_drop = list(np.random.choice(list(purpose), n_trips, replace=False))
                    # Adjust time stamp of prior activity and charging demand of post activity
                    for i in to_drop:
                        profile.loc[i-2, 'charge_time'] += profile.loc[i, 'park_end'] - profile.loc[i-2, 'park_end']
                        profile.loc[i-2, 'park_end'] = profile.loc[i, 'park_end']
                        # Substract drive consumption, if not charging at this activity
                        if profile.loc[i, 'chargingdemand'] == 0:
                            j = i
                            while j < len(profile)-2:
                                # Substract demand here
                                if profile.loc[j+2, 'chargingdemand'] > 0:
                                    profile.loc[j+2, 'chargingdemand'] -= profile.loc[i-1, 'consumption']
                                    break
                                # Otherwise look into the next activity
                                else:
                                    j +=2
                    # Also drop the trip to this location
                    to_drop += [i-1 for i in to_drop]
                    profile.drop(to_drop).reset_index(drop=True, inplace=True)
                
                # Reduce the driving distance (charging demand) by the difference in average distance
                # Correct rounding issues from above
                if n_trips <= n_trips_ref:
                    correction = n_trips_ref * car_use_diff['volume'] / max(1, n_trips)
                else:
                    correction = car_use_diff['volume']
                purpose = list(profile.loc[profile['location']==p_simbev].index)
                for i in purpose:
                    # If a charger is available at this activity
                    if profile.loc[i, 'chargingdemand'] > 0:
                        profile.loc[i, 'chargingdemand'] *= car_use_diff['pkm'] * correction
                    # Otherwise charge at the next activity with charger
                    else:
                        j = i
                        while j < len(profile)-2:
                            if profile.loc[j+2, 'chargingdemand'] > 0:
                                profile.loc[j+2, 'chargingdemand'] *= car_use_diff['pkm'] * correction
                                break
                            else:
                                j +=2
                    # Reduce drive consumption
                    profile.loc[i-1, 'consumption'] *= car_use_diff['pkm'] * correction
                    if i+1 in list(profile.index):
                        profile.loc[i+1, 'consumption'] *= car_use_diff['pkm'] * correction
                # Adjust SOC?
                
            # Aggregate neighboring home stays
            profile.reset_index(drop=True, inplace=True)
            i = len(profile)-1
            while i >= 2:
                if profile.loc[i, 'location'] == '6_home' and profile.loc[i-2, 'location'] == '6_home':
                    # Adjust time stamp of remaining activity
                    profile.loc[i-2, 'charge_time'] += profile.loc[i, 'park_end'] - profile.loc[i-2, 'park_end']
                    profile.loc[i-2, 'park_end'] = profile.loc[i, 'park_end']
                    # Add the charging demand minus drive consumption
                    profile.loc[i-2, 'chargingdemand'] = \
                        profile.loc[i, 'chargingdemand'] + profile.loc[i-2, 'chargingdemand'] \
                        - profile.loc[i-1, 'consumption']
                    # Drop the second activity with its access drive
                    profile.drop([i, i-1], inplace=True)
                    i -= 2
                else:
                    i -= 1
            
            # Prevent negative charging demand
            profile['chargingdemand'].clip(lower=0, inplace=True)
            
            
            # Save modified profile
            profile.reset_index(drop=True).to_csv(path + file)
            charging_demand.append(profile['chargingdemand'].sum())
        
        print('Aggregated charging demand in {}, u={}: {}'.format(scenario, u, round(sum(charging_demand))))

Aggregated charging demand in Avoid, u=2: 3323
Aggregated charging demand in Avoid, u=3: 4414
Aggregated charging demand in Shift, u=2: 4146
Aggregated charging demand in Shift, u=3: 5379
Aggregated charging demand in Avoid+Shift, u=2: 1484
Aggregated charging demand in Avoid+Shift, u=3: 2401


In [12]:
# Reference charging demand
for u in dirs.keys():
    path = scenario_ref+'/'+dirs[u]+'/'
    charging_demand = []
    for file in os.listdir(path):
        profile = pd.read_csv(path + file, index_col=0).reset_index(drop=True)
        charging_demand.append(profile['chargingdemand'].sum())
    print('Aggregated charging demand in {}, u={}: {}'.format(scenario_ref, u, round(sum(charging_demand))))

Aggregated charging demand in reference, u=2: 15380
Aggregated charging demand in reference, u=3: 15952
